In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import make_date, when, col, concat_ws
from pyspark.sql.types import ArrayType
import os
from pyspark.sql.functions import explode

In [2]:
spark = SparkSession.builder.appName("imdb").getOrCreate()

In [3]:
spark

In [ ]:
base_path = os.path.abspath(os.path.join("../.."))
file_path = os.path.join(
    base_path,
    "dataset",
    "imdb",
    "movies",
    "trending_000655.jsonl"
)
df = spark.read.json(file_path)
df.select("raw_payload").printSchema()

root
 |-- raw_payload: struct (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- duration_seconds: long (nullable = true)
 |    |-- genres: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- imdb_id: string (nullable = true)
 |    |-- rating: double (nullable = true)
 |    |-- releaseDate: struct (nullable = true)
 |    |    |-- day: long (nullable = true)
 |    |    |-- month: long (nullable = true)
 |    |    |-- year: long (nullable = true)
 |    |-- title: string (nullable = true)
 |    |-- vote_count: long (nullable = true)



In [40]:
data = df.select("raw_payload.*")
data = data.withColumn(
    "releaseDate",
    make_date(
        col("releaseDate.year"),
        col("releaseDate.month"),
        col("releaseDate.day")
    )
)

# explode arrays
for field in data.schema.fields:
    if isinstance(field.dataType, ArrayType):
        data = data.withColumn(field.name, explode(field.name))
        
data.show()


+--------------------+----------------+-------+----------+------+-----------+--------+----------+
|         description|duration_seconds| genres|   imdb_id|rating|releaseDate|   title|vote_count|
+--------------------+----------------+-------+----------+------+-----------+--------+----------+
|When a new Ghostf...|            6840| Horror|tt27047903|   5.9| 2026-02-27|Scream 7|     24773|
|When a new Ghostf...|            6840|Mystery|tt27047903|   5.9| 2026-02-27|Scream 7|     24773|
+--------------------+----------------+-------+----------+------+-----------+--------+----------+

